In [ ]:
import sys
sys.path.append('..')
from WaveNewmarkFractional import *
from scipy.integrate import simps
from scipy.linalg import eigh, solve_triangular, inv
import pickle
import gc

from utils import *
import numpy as np
from scipy.optimize import minimize
%reload_ext autoreload
%autoreload 2

In [ ]:
mesh = fn.RectangleMesh(fn.Point(-1, -1), fn.Point(1, 1), 50, 50)
fn.plot(mesh, title='Mesh')
V = fn.FunctionSpace(mesh, 'CG', 1)
print('Degree of freedom: ', V.dim())
u_trial, u_test = fn.TrialFunction(V), fn.TestFunction(V)
M_matrix = fn.assemble(u_trial * u_test * fn.dx)
fn.plot(mesh)

In [ ]:
# Setting of the problem:
c = 300.    # Sound speed.

# Define simulation times:
T = 0.2
dt = 0.0005
simulation_times = np.arange(0., T + 0.5 * dt, dt)
# simulation_times = np.array([1., 2., 3.])

In [ ]:
# Computation of eigenfunctions:
# Define priors:
gamma = 1.
delta = 8.
L_form = fn.inner(fn.grad(u_trial), fn.grad(u_test)) * fn.dx
M_form = fn.inner(u_trial, u_test) * fn.dx
R_matrix = fn.assemble(gamma * L_form + delta * M_form)
R_array = R_matrix.array()
M_array = M_matrix.array()
eigvals_R, eigvecs_R = eigh(R_array, M_array)

In [ ]:
def comp_FIM(multi_solution_1, multi_solution_2):
    n = len(multi_solution_1)
    FIM = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            FIM[i, j] = integrate_time_vectors(multi_solution_1[i], multi_solution_2[j], B_matrix, simulation_times)
            FIM[j, i] = FIM[i, j]
    return FIM

def comp_1(observations):
    result = []
    for i in range(len(observations)):
        for j in range(i, len(observations)):
            print('Compute FIM:', i, j)
            temp_result = comp_FIM(observations[i], observations[j])
            result.append(temp_result)
    return result

def comp_2(observations):
    result = []
    for i in range(len(observations)):
        for j in range(len(observations)):
            print('Compute FIM:', i, j)
            temp_result = comp_FIM(observations[i], observations[j])
            result.append(temp_result)
    return result

In [ ]:
ax = plt.figure()
plt.semilogy(1/eigvals_R)
plt.xlabel('Index', fontsize=14)
plt.ylabel('Eigenvalue', fontsize=14)
plt.title('Eigenvalues of $\Gamma_{pr}$', fontsize=14)
plt.axhline(y=2*1e-3, color='r', linestyle='-')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
# Export plot to pdf with short marginal:
# plt.savefig('eigenvalues_laplacian.pdf', bbox_inches='tight')

In [ ]:
list_vectors = [init_vector_numpy(eigvecs_R[:, i]) for i in range(V.dim())]
omega = np.array([100 * np.pi, 150 * np.pi, 200 * np.pi, 300 * np.pi, 400 * np.pi, 500 * np.pi])
size_omega = len(omega)
list_i_funcs = np.array([np.sin(omega[i] * simulation_times) for i in range(size_omega)])
list_i_prime_funcs = np.array([omega[i] * np.cos(omega[i] * simulation_times) for i in range(size_omega)])

In [ ]:
L2_norm_list = np.array([simps(func * func, simulation_times) for func in list_i_funcs])
H1_norm_list = np.array([simps(func * func, simulation_times) for func in list_i_prime_funcs])
print('L2 norm of the functions: ', L2_norm_list)
print('H1 norm of the functions: ', H1_norm_list)

In [ ]:
# Define reference input:
I_0 = 400.
X = np.array([400., 0., 0., 0., 0., 0.])
i_func_ref = list_i_funcs.T.dot(X)
i_prime_func_ref = list_i_prime_funcs.T.dot(X)

# Compute L2 norm of the reference input:
L2_norm_ref = simps(i_func_ref * i_func_ref, simulation_times)
H1_norm_ref = simps(i_prime_func_ref * i_prime_func_ref, simulation_times)
print('L2 norm of the reference function: ', L2_norm_ref)
print('H1 norm of the reference function: ', H1_norm_ref)

In [ ]:
# Plot the reference input:
plt.plot(simulation_times, i_func_ref, label='i')
# plt.plot(simulation_times, i_prime_func_ref, label='i_prime')
plt.xlabel('Time', fontsize=14)
plt.ylabel('Amplitude', fontsize=14)

In [ ]:
# Number of eigenfunctions:
n = 200

# Parameters:
c_expr = fn.Constant(c)
alpha = 0.3 # alpha = 0.3, 0.8
b = (-(2 * c) / np.cos(np.pi * (alpha + 1)/ 2) )
print(b)
# b = 0.0631 # b = 0.1321, 0.0631
b = 0.1321
parameter = [c_expr, b, alpha]

In [ ]:
class RectangleExpression(fn.UserExpression):
    def eval(self, value, x):
        # Define the boundaries of the smaller L-shape within the square [-0.8, 0.8] x [-0.8, 0.8]
        if x[0] >= -0.55 - fn.DOLFIN_EPS and x[0] <= 0.25 + fn.DOLFIN_EPS and x[1] >= -0.4 - fn.DOLFIN_EPS and x[1] <= 0.125 + fn.DOLFIN_EPS:
            value[0] = 3.0  # Value inside the smaller L-shape
        # Define a circle with radius 0.5 and center (0.1, -0.1)
        # elif (x[0] - 0.25)**2 + (x[1] + 0.125)**2 <= 0.3**2:
        #     value[0] = 3.0  # Value inside the circle
        else:
            value[0] = 0.0  # Value outside the smaller L-shape

    def value_shape(self):
        return ()

ic_expr = RectangleExpression(element=V.ufl_element())
u_init = fn.interpolate(ic_expr, V).vector()

# Forward solver:
zero_vec = init_vector_numpy(np.zeros(V.dim()))
f_true = space_time_mult(i_prime_func_ref, u_init, M_matrix, simulation_times)
u_time, _ = FractionalWaveSolverNewmark(V, simulation_times, [zero_vec, zero_vec, f_true], parameter)

In [ ]:
# Observation operator:
class Boundary(fn.SubDomain):
    def inside(self, x, on_boundary):
        left = x[0] < 0.75 + fn.DOLFIN_EPS and x[0] > -0.75 - fn.DOLFIN_EPS
        right = x[1] < 0.75 + fn.DOLFIN_EPS and x[1] > -0.75 - fn.DOLFIN_EPS
        return left and right

# Create a MeshFunction to mark boundaries
subdomains = fn.MeshFunction('size_t', mesh, mesh.topology().dim(), 0)
boundary = Boundary()
boundary.mark(subdomains, 1)  # Mark the left subdomain with the value 1

# Create a measure for the boundary
dx = fn.Measure('dx', domain=mesh, subdomain_data=subdomains)
ds = comp_dS(mesh, subdomains)

u_trial, u_test = fn.TrialFunction(V), fn.TestFunction(V)
B_matrix = fn.assemble(fn.inner(u_trial('+'), u_test('+')) * ds(1))
_misfit = WaveSpaceTimeStateObservation(V, simulation_times, simulation_times, B_matrix)
# Save observations:
p_obs = _misfit.observe(u_time)

In [ ]:
# Save data:
k = 200
multi_solution_100 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[0])
list_100 = export_list_time_vectors('multi_100_08.pkl', multi_solution_100)
del multi_solution_100
gc.collect()

multi_solution_150 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[1])
list_150 = export_list_time_vectors('multi_150_08.pkl', multi_solution_150)
del multi_solution_150
gc.collect()

multi_solution_200 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[2])
list_200 = export_list_time_vectors('multi_200_08.pkl', multi_solution_200)
del multi_solution_200
gc.collect()

multi_solution_300 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[3])
list_300 = export_list_time_vectors('multi_300_08.pkl', multi_solution_300)
del multi_solution_300
gc.collect()

multi_solution_400 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[4])
list_400 = export_list_time_vectors('multi_400_08.pkl', multi_solution_400)
del multi_solution_400
gc.collect()

multi_solution_500 = MultipleFractionalWaveSolver(V, simulation_times, list_vectors[:k], parameter, list_i_prime_funcs[5])
list_500 = export_list_time_vectors('multi_500_08.pkl', multi_solution_500)
del multi_solution_500
gc.collect()

In [ ]:
# Load data:
multi_solution_100 = import_list_time_vectors('data/multi_100_03.pkl', 200, simulation_times, M_matrix)
multi_solution_200 = import_list_time_vectors('data/multi_200_03.pkl', 200, simulation_times, M_matrix)
multi_solution_300 = import_list_time_vectors('data/multi_300_03.pkl', 200, simulation_times, M_matrix)
multi_solution_400 = import_list_time_vectors('data/multi_400_03.pkl', 200, simulation_times, M_matrix)
multi_solution_500 = import_list_time_vectors('data/multi_500_03.pkl', 200, simulation_times, M_matrix)

In [ ]:
k = 160
list_FIM_5 = comp_2([multi_solution_100[:k], multi_solution_200[:k], multi_solution_300[:k], 
                     multi_solution_400[:k], multi_solution_500[:k]])
# Export the list of FIM to a file:
np.savez('data/list_FIM_100_200_300_400_500_03.npz', *list_FIM_5)

In [ ]:
list_FIM_5 = np.load('data/list_FIM_100_200_300_400_500_03.npz')
list_FIM_5 = [list_FIM_5[key] for key in list_FIM_5]
variance = 0.01
k = 160
def OED_func_5(x):
    FIM_sum_0 = list_FIM_5[0]
    FIM_sum_1 = list_FIM_5[6]
    FIM_sum_2 = list_FIM_5[12]
    FIM_sum_3 = list_FIM_5[18]
    FIM_sum_4 = list_FIM_5[24]
    FIM_sum_01 = (list_FIM_5[1] + list_FIM_5[5]) / 2
    FIM_sum_02 = (list_FIM_5[2] + list_FIM_5[10]) / 2
    FIM_sum_03 = (list_FIM_5[3] + list_FIM_5[15]) / 2
    FIM_sum_04 = (list_FIM_5[4] + list_FIM_5[20]) / 2
    FIM_sum_12 = (list_FIM_5[7] + list_FIM_5[11]) / 2
    FIM_sum_13 = (list_FIM_5[8] + list_FIM_5[16]) / 2
    FIM_sum_14 = (list_FIM_5[9] + list_FIM_5[21]) / 2
    FIM_sum_23 = (list_FIM_5[13] + list_FIM_5[17]) / 2
    FIM_sum_24 = (list_FIM_5[14] + list_FIM_5[22]) / 2
    FIM_sum_34 = (list_FIM_5[19] + list_FIM_5[23]) / 2
    FIM = (x[0] ** 2 * FIM_sum_0 + x[1] ** 2 * FIM_sum_1 + x[2] ** 2 * FIM_sum_2 + x[3] ** 2 * FIM_sum_3 + x[4] ** 2 * FIM_sum_4
           + 2 * x[0] * x[1] * FIM_sum_01 + 2 * x[0] * x[2] * FIM_sum_02 + 2 * x[0] * x[3] * FIM_sum_03 + 2 * x[0] * x[4] * FIM_sum_04
           + 2 * x[1] * x[2] * FIM_sum_12 + 2 * x[1] * x[3] * FIM_sum_13 + 2 * x[1] * x[4] * FIM_sum_14
           + 2 * x[2] * x[3] * FIM_sum_23 + 2 * x[2] * x[4] * FIM_sum_24 + 2 * x[3] * x[4] * FIM_sum_34) / variance
    prior_mat = np.diag(eigvals_R[:k])
    post_mat = FIM + prior_mat
    # Check if post_mat is symmetric:
    assert np.allclose(post_mat, post_mat.T), 'Matrix is not symmetric'
    # Check if post_mat is positive definite:
    assert np.all(np.linalg.eigvals(post_mat) > 0), 'Matrix is not positive definite'
    FIM_inv = inv(post_mat)
    FIM_inv_square = FIM_inv @ FIM_inv
    obj = np.trace(FIM_inv)
    grad_0 = -2 * np.trace(FIM_inv_square @ (x[0] * FIM_sum_0 + x[1] * FIM_sum_01 + x[2] * FIM_sum_02 + x[3] * FIM_sum_03 + x[4] * FIM_sum_04)) / variance
    grad_1 = -2 * np.trace(FIM_inv_square @ (x[0] * FIM_sum_01 + x[1] * FIM_sum_1 + x[2] * FIM_sum_12 + x[3] * FIM_sum_13 + x[4] * FIM_sum_14)) / variance
    grad_2 = -2 * np.trace(FIM_inv_square @ (x[0] * FIM_sum_02 + x[1] * FIM_sum_12 + x[2] * FIM_sum_2 + x[3] * FIM_sum_23 + x[4] * FIM_sum_24)) / variance
    grad_3 = -2 * np.trace(FIM_inv_square @ (x[0] * FIM_sum_03 + x[1] * FIM_sum_13 + x[2] * FIM_sum_23 + x[3] * FIM_sum_3 + x[4] * FIM_sum_34)) / variance
    grad_4 = -2 * np.trace(FIM_inv_square @ (x[0] * FIM_sum_04 + x[1] * FIM_sum_14 + x[2] * FIM_sum_24 + x[3] * FIM_sum_34 + x[4] * FIM_sum_4)) / variance
    return obj, np.array([grad_0, grad_1, grad_2, grad_3, grad_4])

In [ ]:
supp = np.array([0, 2, 3, 4, 5])
# def constraint_1(x):
#     return 400. - np.sum(np.abs(x))

def constraint_1(x):
    return 400. - np.sum(np.abs(x))

def constraint_2(x):
    L2_norm = L2_norm_list[supp]
    H1_norm = H1_norm_list[supp]
    return L2_norm_ref - np.sum(L2_norm * x ** 2) + H1_norm_ref - np.sum(H1_norm * x ** 2)

constraints = [{'type': 'ineq', 'fun': constraint_1},
               {'type': 'ineq', 'fun': constraint_2}]

# OED_opt = lambda x: OED_func_4(x, list_FIM_4, eigvals_R, k)
    
x0 = np.array([100., 0., 0., 0., 0])
# Increase number of iterations:
result_scipy = minimize(OED_func_5, x0, jac=True, constraints=constraints, options={'maxiter': 400}, tol=1e-8)
print(result_scipy)

In [ ]:
print(OED_func_5(result_scipy.x))
print(OED_func_5(x0))
print(OED_func_5([0., 0., 0., 0., 80.]))
print(OED_func_5([0., 0., 0., 0., 0.]))

In [ ]:
X = np.sqrt((L2_norm_ref + H1_norm_ref)/(L2_norm_list[-1] + H1_norm_list[-1]))
print(X/400)